# ReadmitIQ — Phase 1: Data understanding

**Question:** Can this publicly available dataset support a reproducible readmission-prioritization project?

This executed report inspects the unfiltered original UCI file. It stops before EDA, cleaning, feature engineering or modeling. This is analytical decision support, not a clinical diagnostic tool.

[Dataset selection](../docs/dataset_selection.md) · [Source dictionary](../docs/data_dictionary.md)

## 1. Verify the source and interpreter

### Question
Are we inspecting the reviewed source bytes with the project environment?

### Analysis

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from readmit_iq.config import load_config
from readmit_iq.data.download import verify_file
from readmit_iq.data.load_data import load_id_mapping, load_raw
from readmit_iq.data.inspect import column_profile, summarize
from readmit_iq.data.validate import validate_raw

config = load_config()
assert sys.prefix != sys.base_prefix
assert Path(sys.prefix).resolve() == config.raw_dir.parent.parent / ".venv"
for filename, checksum in config.dataset.file_sha256.items():
    verify_file(config.raw_dir / filename, checksum)
raw = load_raw(config.raw_dir / "diabetic_data.csv")
mappings = load_id_mapping(config.raw_dir / "IDS_mapping.csv")
validation = validate_raw(raw, config, mappings)
assert validation["passed"], validation["errors"]
profile = column_profile(raw)
summary = summarize(raw, profile, validation)
display({"shape": raw.shape, "Python": sys.version.split()[0], "source_verified": True})

{'shape': (101766, 50), 'Python': '3.12.2', 'source_verified': True}

### Finding
The original source file loads as 101,766 rows and 50 columns. Source checksums and the raw contract pass.

### Decision
Use this immutable raw release. Preserve both identifiers for audit/grouping and all source tokens. The 47 remaining candidate features have not undergone feature selection.

## 2. Inspect structure and repeated patients

### Question
What does a row represent, and are rows independent patients?

### Analysis

In [2]:
display(raw.drop(columns=["encounter_id", "patient_nbr"]).head(3).T.rename_axis("raw_column"))
display(pd.Series({key: summary[key] for key in ["unique_patients", "patients_with_multiple_encounters", "encounters_from_repeat_patients", "additional_encounters_beyond_one_per_patient", "max_encounters_per_patient"]}, name="observed"))
display(profile[["column", "loaded_dtype", "unique_values_including_markers", "min", "max"]])

,0,1,2
raw_column,,,
race,Caucasian,Caucasian,AfricanAmerican
gender,Female,Female,Female
age,[0-10),[10-20),[20-30)
weight,?,?,?
admission_type_id,6,1,1
discharge_disposition_id,25,1,1
admission_source_id,1,7,7
time_in_hospital,1,3,2
payer_code,?,?,?


unique_patients                                 71518
patients_with_multiple_encounters               16773
encounters_from_repeat_patients                 47021
additional_encounters_beyond_one_per_patient    30248
max_encounters_per_patient                         40
Name: observed, dtype: int64

,column,loaded_dtype,unique_values_including_markers,min,max
0,encounter_id,string,101766,NaN,NaN
1,patient_nbr,string,71518,NaN,NaN
2,race,str,6,NaN,NaN
3,gender,str,3,NaN,NaN
4,age,str,10,NaN,NaN
5,weight,str,10,NaN,NaN
6,admission_type_id,int64,8,1.0,8.0
7,discharge_disposition_id,int64,26,1.0,28.0
8,admission_source_id,int64,17,1.0,25.0
9,time_in_hospital,int64,14,1.0,14.0


### Finding
71,518 unique patients contribute 101,766 encounters. There are 16,773 repeat patients and 47,021 encounters belonging to repeat patients. No duplicate encounter IDs or exact duplicate rows were detected.

### Decision
Treat the unit as an encounter, not an independent patient. Design patient-grouped evaluation later. No train/test split is created. There are no encounter timestamps or hospital identifiers; temporal or hospital holdouts cannot be reconstructed from these columns.

## 3. Check the target labels

### Question
How common is the supplied early-readmission label?

### Analysis

In [3]:
counts = raw["readmitted"].value_counts().reindex(["<30", ">30", "NO"])
display(pd.DataFrame({"encounters": counts, "percent": (100 * counts / len(raw)).round(2)}))
display({"positive_rate": summary["positive_rate"], "negative_to_positive_ratio": summary["negative_to_positive_ratio"]})

,encounters,percent
readmitted,,
<30,11357,11.16
>30,35545,34.93
NO,54864,53.91


{'positive_rate': 0.11159915885462728,
 'negative_to_positive_ratio': 7.960641014352381}

### Finding
The `<30` label occurs 11,357 times (11.16%). `>30` occurs 35,545 times and `NO` occurs 54,864 times.

### Decision
Use `<30` as the intended positive label, with `>30` and `NO` as negatives. Unknown labels must fail validation. The day-30 boundary cannot be independently verified. Class imbalance is documented; resampling and class weighting are deferred. These are outcome counts, not fitted model metrics.

## 4. Separate missing markers from unmeasured tests

### Question
Which values are unavailable, and could CSV parsing lose meaningful information?

### Analysis

In [4]:
display(profile.loc[profile["question_mark_count"] > 0, ["column", "question_mark_count", "missing_marker_pct"]].sort_values("question_mark_count", ascending=False))
display(pd.DataFrame({column: raw[column].value_counts() for column in ["A1Cresult", "max_glu_serum"]}).fillna(0).astype(int))
display({"parser_nulls": summary["native_null_total"], "blank_cells": summary["blank_total"], "coded_unknowns": validation["coded_unknowns"]})

,column,question_mark_count,missing_marker_pct
5,weight,98569,96.8585
11,medical_specialty,49949,49.0822
10,payer_code,40256,39.5574
2,race,2273,2.2336
20,diag_3,1423,1.3983
19,diag_2,358,0.3518
18,diag_1,21,0.0206


,A1Cresult,max_glu_serum
>200,0,1485
>300,0,1264
>7,3812,0
>8,8216,0
None,84748,96420
Norm,4990,2597


{'parser_nulls': 0,
 'blank_cells': 0,
 'coded_unknowns': {'admission_type_id': {'5': {'description': 'Not Available',
    'count': 4785},
   '6': {'description': 'NULL', 'count': 5291},
   '8': {'description': 'Not Mapped', 'count': 320}},
  'discharge_disposition_id': {'18': {'description': 'NULL', 'count': 3691},
   '25': {'description': 'Not Mapped', 'count': 989},
   '26': {'description': 'Unknown/Invalid', 'count': 0}},
  'admission_source_id': {'9': {'description': 'Not Available', 'count': 125},
   '15': {'description': 'Not Available', 'count': 0},
   '17': {'description': 'NULL', 'count': 6781},
   '20': {'description': 'Not Mapped', 'count': 161},
   '21': {'description': 'Unknown/Invalid', 'count': 0}}}}

### Finding
Weight is 96.86% missing, medical specialty 49.08%, payer code 39.56%. Other `?` values appear in race and three diagnosis columns. A1c is not measured in 84,748 encounters; glucose is not measured in 96,420. Numeric administrative codes also encode unavailable information.

### Decision
Keep literal `None` as the documented not-measured category. Do not blanket-impute it with unknown values. Defer missingness mechanisms and drop/imputation decisions to later analysis. All original values remain intact.

## 5. Review anomalies and cohort eligibility

### Question
What must be resolved before EDA and model development?

### Analysis

In [5]:
display({"validation_passed": validation["passed"], "invalid_numeric_counts": validation["invalid_numeric_counts"], "unmapped_codes": validation["unmapped_codes"], "constant_columns": summary["constant_columns"], "unknown_invalid_gender": validation["unknown_invalid_gender"]})
display(pd.DataFrame(validation["death_or_hospice_dispositions"]).T)
assert len(raw) == 101_766
assert not list((config.raw_dir.parent / "processed").glob("*.csv"))

{'validation_passed': True,
 'invalid_numeric_counts': {'time_in_hospital': 0,
  'num_lab_procedures': 0,
  'num_procedures': 0,
  'num_medications': 0,
  'number_outpatient': 0,
  'number_emergency': 0,
  'number_inpatient': 0,
  'number_diagnoses': 0},
 'unmapped_codes': {'admission_type_id': [],
  'discharge_disposition_id': [],
  'admission_source_id': []},
 'constant_columns': ['examide', 'citoglipton'],
 'unknown_invalid_gender': 3}

,description,count
11,Expired,1642
13,Hospice / home,399
14,Hospice / medical facility,372
19,"Expired at home. Medicaid only, hospice.",8
20,"Expired in a medical facility. Medicaid only, ...",2
21,"Expired, place unknown. Medicaid only, hospice.",0


### Finding
`examide` and `citoglipton` are constant. Three gender records are `Unknown/Invalid`. Death/hospice dispositions account for 2,423 encounters. Source count bounds and code-map checks found no violations. This does not certify every record as clinically plausible.

### Decision
Review eligibility for discharge outreach before cohort filtering. Review availability of discharge and within-stay fields at the proposed discharge prediction time. Preserve demographics for later responsible-ML assessment; no fairness results are claimed.

## Phase 1 stopping point

Raw ingestion and inspection are complete. No EDA charts, imputation, cohort exclusions, engineered features, fitted preprocessing, data splits, training or API have been produced.

Next: review these findings and agree on the eligible discharge cohort, then proceed to question-driven EDA.

Sources: [UCI release](https://doi.org/10.24432/C5230J), [original study](https://pmc.ncbi.nlm.nih.gov/articles/PMC3996476/), and checksum-verified local computations.